<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week5_Day4_Daily_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch scikit-learn pandas numpy joblib gensim spacy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import joblib

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import zipfile

zip_path = '/content/archive.zip'
extract_path = '/content/extracted_data'

if os.path.exists(zip_path):
    file_size = os.path.getsize(zip_path)
    print(f"File size: {file_size} bytes")

    if file_size > 0:
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
            print(f"Files successfully extracted to: {extract_path}")
        except zipfile.BadZipFile:
            print("Error: The file exists but is not a valid zip archive.")
    else:
        print("Error: The file is empty (0 bytes).")
else:
    print(f"Error: File not found at {zip_path}")

File size: 67108864 bytes
Error: The file exists but is not a valid zip archive.


In [ ]:
import os

# Using 7z (7-zip) which is often more successful with non-standard or 'streamed' zip files
print("Attempting extraction with 7z...")
extract_path = '/content/extracted_data'
os.makedirs(extract_path, exist_ok=True)

!7z x /content/archive.zip -o{extract_path} -y

print("\nChecking for extracted files:")
found = False
for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(os.path.join(root, file))
        found = True

if not found:
    print("7z also failed to extract any files. The archive may be part of a split set or severely truncated.")

Attempting extraction with 7z...

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/                   1 file, 67108864 bytes (64 MiB)

Extracting archive: /content/archive.zip
 11% 256 Open             
ERRORS:
Unexpected end of archive

--
Path = /content/archive.zip
Type = zip
ERRORS:
Unexpected end of archive
Physical Size = 67129353

 19%     10% 228 - etfs/CORN.csv                         20% 382         29% 513 - etfs/EWS.csv                        37% 659 - etfs/FMF.csv                        45% 780 - etfs/GOVT.csv                    

In [ ]:
import os
import pandas as pd

extracted_dir = '/content/extracted_data/etfs'
if os.path.exists(extracted_dir):
    all_files = sorted(os.listdir(extracted_dir))
    print(f"Total files extracted: {len(all_files)}")
    print(f"First 10 files: {all_files[:10]}")

    # Load a sample file to verify content
    sample_file = os.path.join(extracted_dir, all_files[0])
    print(f"\nPreviewing {all_files[0]}:")
    df_sample = pd.read_csv(sample_file)
    display(df_sample.head())
else:
    print("Extraction directory not found.")

Total files extracted: 1677
First 10 files: ['AAAU.csv', 'AADR.csv', 'AAXJ.csv', 'ABEQ.csv', 'ACES.csv', 'ACIO.csv', 'ACSG.csv', 'ACSI.csv', 'ACT.csv', 'ACWF.csv']

Previewing AAAU.csv:


,Date,Open,High,Low,Close,Adj Close,Volume
0,2018-08-15,11.84,11.84,11.74,11.74,11.74,27300
1,2018-08-16,11.78,11.80,11.74,11.74,11.74,428400
2,2018-08-17,11.80,11.82,11.77,11.82,11.82,52400
3,2018-08-20,11.88,11.91,11.85,11.90,11.90,28700
4,2018-08-21,11.92,11.95,11.89,11.93,11.93,30600


In [ ]:
print(df.shape)

print(df.info())

df.head()

In [17]:
# On définit df à partir de l'échantillon chargé précédemment
df = df_sample

# Supprimer les colonnes inutiles
columns_to_drop = []

for col in ['Date', 'Name', 'Symbol']:
    if col in df.columns:
        columns_to_drop.append(col)

df.drop(columns=columns_to_drop, inplace=True)


# Créer la variable cible : prix de fermeture du lendemain
df['Target'] = df['Close'].shift(-1)

# Supprimer la dernière ligne vide
df.dropna(inplace=True)

df.head()

,Open,High,Low,Close,Adj Close,Volume,Target
0,11.84,11.84,11.74,11.74,11.74,27300,11.74
1,11.78,11.80,11.74,11.74,11.74,428400,11.82
2,11.80,11.82,11.77,11.82,11.82,52400,11.90
3,11.88,11.91,11.85,11.90,11.90,28700,11.93
4,11.92,11.95,11.89,11.93,11.93,30600,11.96


In [18]:
scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(df)

scaled_df = pd.DataFrame(
    scaled_data,
    columns=df.columns
)

scaled_df.head()

,Open,High,Low,Close,Adj Close,Volume,Target
0,0.011929,0.007921,0.000000,0.000000,0.000000,0.012286,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.193460,0.016032
2,0.003976,0.003960,0.006198,0.016032,0.016032,0.023623,0.032064
3,0.019881,0.021782,0.022727,0.032064,0.032064,0.012918,0.038076
4,0.027833,0.029703,0.030992,0.038076,0.038076,0.013777,0.044088


In [19]:
sequence_length = 30

X = []
y = []

data = scaled_df.values

for i in range(len(data) - sequence_length):

    X.append(
        data[i:i + sequence_length, :-1]
    )

    y.append(
        data[i + sequence_length, -1]
    )

X = np.array(X)
y = np.array(y)

print("X shape :", X.shape)
print("y shape :", y.shape)

X shape : (379, 30, 6)
y shape : (379,)


In [20]:
total_size = len(X)

train_size = int(total_size * 0.7)
val_size = int(total_size * 0.15)


X_train = X[:train_size]
y_train = y[:train_size]


X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]


X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]


print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (265, 30, 6)
Validation: (56, 30, 6)
Test: (58, 30, 6)


In [21]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [22]:
batch_size = 32

train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [23]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

In [24]:
input_size = X_train.shape[2]

model = LSTMModel(input_size=input_size)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [26]:
epochs = 20

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1/20 | Train Loss: 0.0165 | Val Loss: 0.0041
Epoch 2/20 | Train Loss: 0.0150 | Val Loss: 0.0038
Epoch 3/20 | Train Loss: 0.0156 | Val Loss: 0.0041
Epoch 4/20 | Train Loss: 0.0140 | Val Loss: 0.0038
Epoch 5/20 | Train Loss: 0.0149 | Val Loss: 0.0037
Epoch 6/20 | Train Loss: 0.0142 | Val Loss: 0.0037
Epoch 7/20 | Train Loss: 0.0150 | Val Loss: 0.0038
Epoch 8/20 | Train Loss: 0.0142 | Val Loss: 0.0037
Epoch 9/20 | Train Loss: 0.0143 | Val Loss: 0.0039
Epoch 10/20 | Train Loss: 0.0126 | Val Loss: 0.0037
Epoch 11/20 | Train Loss: 0.0136 | Val Loss: 0.0037
Epoch 12/20 | Train Loss: 0.0134 | Val Loss: 0.0037
Epoch 13/20 | Train Loss: 0.0149 | Val Loss: 0.0039
Epoch 14/20 | Train Loss: 0.0136 | Val Loss: 0.0037
Epoch 15/20 | Train Loss: 0.0144 | Val Loss: 0.0039
Epoch 16/20 | Train Loss: 0.0137 | Val Loss: 0.0037
Epoch 17/20 | Train Loss: 0.0137 | Val Loss: 0.0038
Epoch 18/20 | Train Loss: 0.0131 | Val Loss: 0.0037
Epoch 19/20 | Train Loss: 0.0145 | Val Loss: 0.0038
Epoch 20/20 | Train L

In [27]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        predictions.extend(outputs.numpy())
        actuals.extend(y_batch.numpy())

r2 = r2_score(actuals, predictions)

print("R² Score:", r2)

R² Score: -0.18612184908909524


In [28]:
torch.save(model.state_dict(), "lstm_stock_model.pth")

joblib.dump(scaler, "scaler.pkl")

print("Modèle et scaler sauvegardés !")

Modèle et scaler sauvegardés !


In [29]:
model.eval()

sample = torch.tensor(X_test[0:1], dtype=torch.float32)

with torch.no_grad():
    pred = model(sample)

print("Prediction :", pred.item())

Prediction : 0.7140852808952332
